# Topic analysis Citations — saved direct-citation communities

Interpret the **direct-citation Leiden partition already selected in `Clustering PDFs.ipynb`**, at resolution **1.0**, selected seed **43**, including reviewed manual citation links. Its **six communities** contain **289 canonical works**, with **three isolated works unassigned**. Two duplicate PDF aliases are mapped to canonical works, leaving 292 works from the 294 selected PDFs.

This notebook loads saved assignments: it never fits, sweeps, merges or chooses replacement communities. Citation communities are structurally defined by references; their topic names describe their retained PDF source text. They need not be thematically uniform. The tiny C6 is a separate three-work connected component.

The existing snapshot was checked against the source run's graph hashes, authenticated plotted labels, and recorded seed memberships. Future source runs export `citation_selected` directly. Missing or changed inputs stop execution. For the shared reference use the supplied snapshots; rerunning source clustering is unnecessary.


## 1. Validate the canonical works and saved citation partition

The graph uses the binary undirected union of directed citations. Original community IDs C1–C6 are preserved; zero means unassigned, not a seventh topic. Alias PDFs are listed separately and do not contribute additional term counts.


In [1]:
from pathlib import Path
import sys
from datetime import datetime, timezone
from importlib.metadata import version
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'batill').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from batill.citation_topic_analysis import (
    load_citation_topic_corpus, load_citation_topic_partition,
    describe_citation_topics, write_citation_review_cards,
)
from batill.topic_analysis import build_vocabulary, topic_terms, STOP_WORDS
from batill.storage import file_hash, write_json

INPUT_DIR = ROOT / 'outputs/corpus_sections'
EXCLUSIONS_FILE = ROOT / 'configs/analysis_exclusions.json'
GRAPH_DIR = ROOT / 'outputs/citation_graph_openalex/9ad47e071ee12c2f/reviewed'
corpus = load_citation_topic_corpus(INPUT_DIR, EXCLUSIONS_FILE, GRAPH_DIR, include_manual=True)
PARTITION_DIR = (ROOT / 'outputs/citation_topic_partitions'
                 / corpus['selection']['selection_id'][:16] / 'resolution_1')
labels, partition_manifest = load_citation_topic_partition(PARTITION_DIR, corpus)
if partition_manifest['resolution'] != 1.0 or partition_manifest['communities'] != 6:
    raise ValueError('This reference analysis expects the saved resolution-1 six-community solution.')
SOLUTION = 'citation_r1'
RESULTS = (ROOT / 'reports/topic_analysis_citations' / corpus['selection']['selection_id'][:16]
           / partition_manifest['assignments_sha256'][:16])
RESULTS.mkdir(parents=True, exist_ok=True)
corpus['papers'].drop(columns='text').to_csv(RESULTS / 'papers.csv', index=False)
corpus['exclusion_audit'].to_csv(RESULTS / 'exclusion_audit.csv', index=False)
aliases = corpus['mapping'].copy()
aliases['cluster'] = aliases.canonical_key.map(pd.Series(labels, index=corpus['papers'].pdf_sha256))
aliases.to_csv(RESULTS / 'pdf_to_work_memberships.csv', index=False)
write_json(RESULTS / 'source_partition_manifest.json', partition_manifest)
write_json(RESULTS / 'graph_diagnostics.json', corpus['network']['diagnostics'])
display(pd.DataFrame([corpus['network']['diagnostics']]))
print('Saved source resolution:', partition_manifest['resolution'], 'seed:', partition_manifest['seed'])
print('Evidence folder:', RESULTS)


Saved source resolution: 1.0 seed: 43
Evidence folder: /Users/tilldedek/Documents/Semester 6/Bachelorarbeit/Python/batill/reports/topic_analysis_citations/9ad47e071ee12c2f/20563542c693f5e4


,works,directed_citations,undirected_links,clustered_works,isolated_works,components,nonisolated_components,largest_component_works
0,292,2539,2480,289,3,5,2,286


## 2. Compute text evidence within the existing citation communities

Use the exact retained text of each canonical PDF, excluding isolates from both vocabulary and inside/outside comparisons. The vocabulary contains words and contiguous two-word phrases present in at least three works and at most 90% of the 289 assigned works. English stopwords and formatting tokens are removed, without stemming or synonym merging. Bigrams do not bridge punctuation or stopwords.

**c-TF-IDF** weights pooled community word counts. **Per-work TF-IDF contrast** subtracts the outside-community mean from the inside mean. A listed term must occur in at least two works and 10% of the community. These descriptive scores are not significance tests.

Select three representatives by internal citation degree, breaking ties by total incoming citations and then corpus order. Select two boundary candidates by largest fraction of links to other communities and up to two additional random works using seed 42 plus the cluster ID. A disconnected community with no external links has no such boundary candidates. Excerpts are exact retained source text with page, block and character offsets; no generated summaries or embedding silhouettes are used.


In [2]:
active = labels > 0
vocabulary = build_vocabulary(corpus['papers'].loc[active], min_df=3, max_df=.9)
term_scores = topic_terms(vocabulary, labels[active], top_n=20)
profiles, assignments, review_papers = describe_citation_topics(corpus, labels, term_scores)
for frame in (profiles, assignments, review_papers, term_scores):
    frame.insert(0, 'solution', SOLUTION)
write_json(RESULTS / 'vocabulary.json', vocabulary['terms'].tolist())
write_json(RESULTS / 'stopwords.json', sorted(STOP_WORDS))
for frame, filename in [(profiles, 'cluster_profiles'), (assignments, 'memberships'),
                        (review_papers, 'review_papers'), (term_scores, 'term_scores')]:
    frame.to_csv(RESULTS / f'{filename}.csv', index=False)
np.testing.assert_array_equal(assignments.cluster.to_numpy(), labels)
display(profiles[['cluster', 'papers', 'within_community_links', 'contrast_terms']])
print('All canonical-work assignments preserved, including the unassigned isolates.')


All canonical-work assignments preserved, including the unassigned isolates.


,cluster,papers,within_community_links,contrast_terms
0,1,46,233,employment; fe; davis; treatment; equity buyou...
1,2,97,573,lbos; lbo; announcement; mbos; debt; managemen...
2,3,29,53,entrepreneurs; entrepreneurial; entrepreneur; ...
3,4,89,794,pme; vintage; lps; irr; gps; gp; vc; buyout fu...
4,5,25,96,hospital; patients; physician; health care; me...
5,6,3,3,gastroenterology; physician; practice manageme...


## 3. Proposed names and their evidence

Names are **recorded AI-assisted analyst proposals**, based on terms, coverage, titles and sampled source passages. They are not generated by regex, and no language-model call is made when rerunning. Regex is used only for word tokenization and locating excerpts. Names remain proposals pending reviewer approval.

`configs/citation_topic_labels.csv` binds each name to the solution, original community ID, member PDF hashes and exact retained texts. A changed group or text requires a new review. The notebook never overwrites this interpretive input; missing or stale proposals display “Name pending review”. Isolates receive no topic name.


In [3]:
PROPOSAL_FILE = ROOT / 'configs/citation_topic_labels.csv'
if PROPOSAL_FILE.exists():
    proposals = pd.read_csv(PROPOSAL_FILE, keep_default_na=False)
else:
    proposals = pd.DataFrame(columns=['solution', 'cluster', 'membership_id', 'proposed_label', 'status'])
named_profiles = write_citation_review_cards(
    profiles, term_scores, review_papers, proposals, RESULTS / 'cluster_review_cards.md')
named_profiles.to_csv(RESULTS / 'named_cluster_profiles.csv', index=False)
named_assignments = assignments.merge(
    named_profiles[['solution', 'cluster', 'proposed_label', 'proposal_matched']],
    on=['solution', 'cluster'], how='left', validate='many_to_one')
named_assignments.loc[named_assignments.cluster.eq(0), 'proposed_label'] = 'Unassigned (no citation links)'
named_assignments['proposal_matched'] = named_assignments.proposal_matched.fillna(False).astype(bool)
named_assignments.to_csv(RESULTS / 'named_memberships.csv', index=False)
if PROPOSAL_FILE.exists():
    proposals.to_csv(RESULTS / 'label_proposals_snapshot.csv', index=False)
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display(named_profiles[['cluster', 'proposed_label', 'papers', 'proposal_matched']])
    display(named_assignments.loc[named_assignments.cluster.eq(0), ['paper_id', 'query_title', 'proposed_label']])
print('All six names match:', bool(named_profiles.proposal_matched.all()))


All six names match: True


,cluster,proposed_label,papers,proposal_matched
0,1,"Private equity ownership: employment, productivity and real effects",46,True
1,2,"Leveraged buyouts: financing, governance and value creation",97,True
2,3,Venture capital and private equity: entrepreneurial finance and institutions,29,True
3,4,"Private equity funds: performance, risk and investor decisions",89,True
4,5,"Private equity in healthcare: ownership, costs and quality",25,True
5,6,Gastroenterology practices: private equity and physician management,3,True


,paper_id,query_title,proposed_label
35,P036,Size Management by European Private Firms to Minimize Proprietary Costs of Disclosure,Unassigned (no citation links)
120,P125,"Boards, CEO entrenchment, and the cost of capital",Unassigned (no citation links)
289,P315,Institutional liquidity needs and the structure of monitored finance,Unassigned (no citation links)


## 4. Inspect a community

Compare both term rankings, their inside/outside coverage, representative and boundary works, and random excerpts. Review the full membership list before approving a name. Community IDs have no correspondence to the PDF embedding or abstract topic IDs.


In [4]:
INSPECT_CLUSTER = 1  # Original citation IDs 1–6.
if INSPECT_CLUSTER not in set(labels) - {0}:
    raise ValueError('Choose one of the original assigned citation communities.')
display(term_scores.loc[term_scores.cluster.eq(INSPECT_CLUSTER),
    ['ranking', 'rank', 'term', 'inside_count', 'inside_n', 'outside_count', 'outside_n', 'prevalence_gap', 'score']])
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display(review_papers.loc[review_papers.cluster.eq(INSPECT_CLUSTER),
        ['paper_id', 'title', 'selection', 'within_community_degree', 'external_link_fraction',
         'page', 'block_id', 'term', 'excerpt_start', 'excerpt_end', 'excerpt']])
    display(named_assignments.loc[named_assignments.cluster.eq(INSPECT_CLUSTER),
        ['paper_id', 'query_title', 'citation_degree', 'within_community_degree', 'external_link_fraction']])


,ranking,rank,term,inside_count,inside_n,outside_count,outside_n,prevalence_gap,score
0,c_tf_idf,1,pe,41,46,165,243,0.212292,0.049488
1,c_tf_idf,2,yes,31,46,105,243,0.241814,0.032698
2,c_tf_idf,3,yes yes,23,46,79,243,0.174897,0.022705
3,c_tf_idf,4,panel,41,46,190,243,0.109411,0.017652
4,c_tf_idf,5,target,44,46,136,243,0.396851,0.017482
5,c_tf_idf,6,buyout,35,46,199,243,-0.058060,0.017163
6,c_tf_idf,7,buyouts,39,46,179,243,0.111201,0.016270
7,c_tf_idf,8,impact,42,46,197,243,0.102344,0.014144
8,c_tf_idf,9,growth,42,46,204,243,0.073537,0.013864
9,c_tf_idf,10,employment,33,46,61,243,0.466362,0.013603


,paper_id,title,selection,within_community_degree,external_link_fraction,page,block_id,term,excerpt_start,excerpt_end,excerpt
0,P116,"Private equity, jobs, and productivity",representative,31,0.279070,1,/page/0/Footnote/5,employment,762,1362,"Bureau of Economic Research. This paper incorporates material from two earlier working papers by the authors, ""Private Equity and Employment"" (2011) and ""Private Equity, Jobs, and Productivity"" (2012). We thank Chris Allen, Paul Bailey, Ronald Davis, and Sarah Woolverton for research assistance, and Per Stromberg for data on the classification of private equity transactions. Francesca Cornelli, Per Stromberg, three anonymous referees, several practitioners, and numerous participants in conferences and research seminars provided many helpful comments. The World Economic Forum, Kauffman Foundati"
1,P040,The Operational Consequences of Private Equity Buyouts: Evidence from the Restaurant Industry,representative,26,0.434783,3,/page/2/Footnote/5,employment,1,205,"Exceptions are papers that focus on nonfinancial performance margins, such as innovation (Lerner, Sorensen, and Strömberg 2011), employment (Davis et al. 2014), and tax return data (Cohn and Towery 2013)"
2,P053,Growth LBOs,representative,25,0.418605,1,/page/0/Text/31,employment,138,738,"taining operating income constant. Lichtenberg and Siegel (1990) find that privately held LBO targets tend to reduce white-collar employment and wages. More recently, Amess and Wright (2007) and Davis, Haltiwanger, Jarmin, Lerner, and Javier (2008), studying the UK and the US market, over a longer time period, find that initially privately held LBO targets experience some employment reduction following a leveraged buyout. Finally, Chevalier (1995) and Chevalier and Scharfstein (1996) provide evidence that, in the 1980s, LBOs in the supermarket industry led to underinvestment in market shares."
3,P026,Earnings management in private equity portfolio firms during fundraising,boundary,5,0.722222,14,/page/13/Table/7,fe,0,372,Earnings management FundraiseFlag × Low FundraiseFlag Low reputation Controls Country–industry FE Year FE | before PE investment. (1) Abn. Accruals reputation − 0.040 ( − 0.52) − 0.018 ( − 0.65) − 0.037 ( − 1.41) Y Y Y | (2) Disc. Sales 0.024 (0.57) − 0.011 ( − 0.73) 0.016 (0.95) Y Y Y\nN | 1029 | 1168\nR-sq | 0.421 | 0.522\nClustering | Country–industry | Country–industry
4,P117,The (Heterogeneous) Economic Effects of Private Equity Buyouts,boundary,8,0.666667,1,/page/0/Text/7,employment,0,600,"Abstract. The effects of private equity buyouts on employment, productivity, and job reallocation vary tremendously with macroeconomic and credit conditions, across private equity groups, and by type of buyout. We reach this conclusion by examining the most extensive database of U.S. buyouts ever compiled, encompassing thousands of buyout targets from 1980 to 2013 and millions of control firms. Employment shrinks 12% over two years after buyouts of publicly listed firms—on average, and relative to control firms—but expands 15% after buyouts of privately held firms. Postbuyout productivity gain"
5,P184,Do Employees Cheer for Private Equity? The Heterogeneous Effects of Buyouts on Job Quality,random,12,0.520000,3,/page/2/Text/8,employment,273,873,"loyment incidence at LBO targets in Sweden, Fang et al. (2021) on wage gaps at LBO targets in France, and Antoni et al. (2019) on employment and wages at LBO targets in Germany. Our work builds upon this literature in four ways. First, we shed new light on the importance of nonpecuniary amenities, such as firm culture, which are important but largely unstudied (e.g., Lins et al. 2017, Mas and Pallais 2017, Lamadon et al. 2022). Second, our performance pay measure includes stock option grants and other types of variable pay that are typically difficult to observe. Third, we study a large and re"
6,P150,Private Equity and Pay Gaps Inside the Firm,random,11,0.214286,3,/page/2/Text/4,employment,0,600,"W

,paper_id,query_title,citation_degree,within_community_degree,external_link_fraction
0,P001,ESG Disclosures in the Private Equity Industry,9,6,0.333333
7,P008,Private Equity and Workers' Career Paths: The Role of Technological Change,21,16,0.238095
14,P015,Private equity and human capital risk,14,9,0.357143
25,P026,Earnings management in private equity portfolio firms during fundraising,18,5,0.722222
26,P027,Do Public Financial Statements Influence Private Equity and Venture Capital Financing?,6,4,0.333333
33,P034,Does Private Equity Ownership Make Firms Cleaner? the Role of Environmental Liability Risks,15,15,0.000000
36,P037,The Effects of Public and Private Equity Markets on Firm Behavior,19,13,0.315789
37,P038,Private equity and industry performance,28,11,0.607143
38,P039,Private equity and financial fragility during the crisis,26,15,0.423077
39,P040,The Operational Consequences of Private Equity Buyouts: Evidence from the Restaurant Industry,46,26,0.434783


## 5. Record the interpretation run

Record the input and assignment hashes, naming input, code, package versions and artifact checksums. Reproducing the evidence and names requires the saved graph, text, partitions and proposal CSV; it requires no external API, extraction, embedding inference or community detection.


In [5]:
manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'workflow': 'interpret_saved_direct_citation_partition', 'clustering_performed': False,
    'selection': corpus['selection'], 'input_hashes': corpus['input_hashes'],
    'partition_manifest_sha256': file_hash(PARTITION_DIR / 'partition_manifest.json'),
    'source_assignments_sha256': partition_manifest['assignments_sha256'],
    'resolution': partition_manifest['resolution'], 'seed': partition_manifest['seed'],
    'unit': 'canonical_work', 'unassigned_excluded_from_terms': True,
    'config': {'min_df': 3, 'max_df': .9, 'ngrams': [1, 2],
               'minimum_cluster_term_support': 'max(2, ceil(0.1*n))', 'review_random_seed': 42},
    'naming': 'Recorded AI-assisted analyst proposals; not automatic naming or user validation',
    'label_proposals_sha256': file_hash(PROPOSAL_FILE) if PROPOSAL_FILE.exists() else None,
    'all_names_match_memberships': bool(named_profiles.proposal_matched.all()),
    'python': sys.version,
    'packages': {p: version(p) for p in ['numpy', 'pandas', 'scipy', 'scikit-learn']},
    'code_sha256': {p.name: file_hash(p) for p in [ROOT / 'src/batill/citation_topic_analysis.py',
        ROOT / 'src/batill/citation_clustering.py', ROOT / 'src/batill/pdf_topic_analysis.py',
        ROOT / 'src/batill/topic_analysis.py']},
    'artifacts_sha256': {p.name: file_hash(p) for p in RESULTS.iterdir()
                        if p.is_file() and p.name != 'run_manifest.json'},
}
write_json(RESULTS / 'run_manifest.json', manifest)
print('Saved citation topic evidence and names:', RESULTS)


Saved citation topic evidence and names: /Users/tilldedek/Documents/Semester 6/Bachelorarbeit/Python/batill/reports/topic_analysis_citations/9ad47e071ee12c2f/20563542c693f5e4


## Reproduce and share the complete analysis

The supervisor handoff is `outputs/topic-analysis-reproduction.tar.gz`. It contains the exact saved inputs, memberships, naming proposals, code, pinned dependency lock and reference evidence for all three topic notebooks. Sending this notebook alone is insufficient.

After unpacking, use Python 3.12 inside the bundle folder:

```bash
python3.12 -m venv .venv
.venv/bin/python -m pip install -r configs/topic-analysis-requirements.lock
.venv/bin/python scripts/reproduce_topic_analyses.py --check
```

The checker runs each notebook in a fresh process with clustering, extraction, model loading and network calls disabled. It requires exact checksums for the deterministic tables, names, excerpts and review cards. Timestamps, absolute paths in run manifests and notebook display output are excluded from byte comparison. Names are recorded interpretive inputs; reusing their CSV reproduces the names, while rerunning recomputes the supporting evidence. See the README section **Reproduce all three saved-partition topic analyses** for scope, environment limits and update instructions.
